# Practice: CPO and ICE

In the lessons of the module we went through how a model's prediction changes when one feature of an object is changed: the CPO curve for one object, the bundle of ICE curves for a set of objects, and two ICE variants — centered (c-ICE) and derivative (d-ICE). In this notebook all of this is built by hand on real data and checked against the implementation in `scikit-learn`.

Every task has a cell with `...` that you fill in. The self-check quizzes on this notebook are in the lesson “CPO and ICE: Practice” of the module “Explanation Methods for Machine Learning Models pt. 1” on the course platform. Check yourself against the plots and against the comparison with `sklearn` in the cells as well.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence

%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## Data and model

The California Housing dataset ships with `sklearn`: 20,640 California block groups from the 1990 census, eight block-group features, and the target is the median house value in the block group in hundreds of thousands of dollars. The features: `MedInc` — median income (tens of thousands of dollars), `HouseAge` — median house age, `AveRooms` and `AveBedrms` — average number of rooms and bedrooms per house, `Population` — block-group population, `AveOccup` — average number of occupants per house, `Latitude` and `Longitude`.

We split the data into a training and a test part (80% and 20%, `random_state=42`) and train a gradient boosting model `HistGradientBoostingRegressor` with default parameters. For the methods of this notebook the model is a black box: all they need from it is the prediction `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"Test R^2 of the model: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

## 1. CPO: one object, one feature

We fix an object $x_i$, run through the values $z$ of feature $j$ on a grid, and for each value ask for the prediction $f(x_i^{z})$, where $x_i^{z}$ is the object $x_i$ with feature $j$ replaced by $z$.

**Task 1.** Complete the function `cpo_curve`: it copies the object `x` into `len(grid)` rows, puts the grid values into the column `feature` and returns the model's predictions. The grid for the plots is 20 points between the 5th and the 95th percentile of the feature on the training data.

In [ ]:
def cpo_curve(model, x, feature, grid):
    X_rep = ...  # your code: len(grid) copies of x with the grid values in the column feature
    return ...


def quantile_grid(X, feature, k=20):
    return np.linspace(X[feature].quantile(0.05), X[feature].quantile(0.95), k)


x0 = X_test.iloc[0]
f0 = model.predict(X_test.iloc[[0]])[0]
print(f"Model prediction for the object x0: {f0:.3f}")
x0.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, feature in zip(axes, ["MedInc", "Latitude"]):
    grid = quantile_grid(X_train, feature)
    ax.plot(grid, cpo_curve(model, x0, feature, grid), color=BLUE, lw=2)
    ax.scatter([x0[feature]], [f0], color=ORANGE, zorder=3, label="the object x0 itself")
    ax.set_xlabel(feature)
    ax.legend()
axes[0].set_ylabel("prediction, hundreds of thousands of dollars")
fig.suptitle("CPO curves of the object x0: California Housing, gradient boosting")
plt.show()

## 2. The CPO score

$$O_j^{uni}(x_i) = \frac{1}{k}\sum_{l=1}^{k}\bigl| f(x_i^{z_l}) - f(x_i) \bigr|,\qquad O_j^{emp}(x_i) = \frac{1}{n}\sum_{m=1}^{n}\bigl| f(x_i^{z_m}) - f(x_i) \bigr|.$$

In the first score $z_l$ are the points of a uniform grid over the feature's range, in the second $z_m$ are the feature values of $n$ observations of the sample.

**Task 2.** Complete `o_uni` and `o_emp` and compute both scores for the object `x0` over all features. Take the uniform grid of $k=20$ points over the whole range of the feature on the training data, from the minimum to the maximum; the empirical score — over the feature values of 500 random training rows (`random_state=0`).

In [ ]:
def o_uni(model, x, feature, X_ref, k=20):
    grid = ...  # your code: k points from the feature minimum to its maximum
    f_x = model.predict(pd.DataFrame([x]))[0]
    return ...


def o_emp(model, x, feature, X_ref):
    f_x = model.predict(pd.DataFrame([x]))[0]
    return ...  # your code: the mean absolute deviation of the prediction over the feature values in X_ref


X_ref = X_train.sample(500, random_state=0)
cpo_scores = pd.DataFrame({
    "O_uni": [o_uni(model, x0, f, X_train) for f in X.columns],
    "O_emp": [o_emp(model, x0, f, X_ref) for f in X.columns],
}, index=X.columns)
cpo_scores.round(3)

**Space for your conclusions:**
- which feature swings the prediction of the object `x0` most under each of the two scores?
- for which features do the two scores disagree most, and why? Hint: compare the feature's maximum with its 75th percentile in `X.describe()`.

## 3. ICE: a bundle of CPO curves

ICE is CPO curves for a whole set of objects at once. We take 200 random objects of the test sample.

**Task 3.** Complete `ice_curves`: the function returns a matrix of shape `(len(X), len(grid))` where a row is the ICE curve of one object. Do it with a single call to `model.predict`: stack `len(grid)` copies of `X` and in each copy put its own grid value into the column `feature`. Then check the result against `sklearn.inspection.partial_dependence(kind='individual')` on the same grid.

In [ ]:
def ice_curves(model, X, feature, grid):
    X_rep = ...  # your code: len(grid) copies of X stacked; in copy t the column feature equals grid[t]
    return ...   # a (len(X), len(grid)) matrix


X_ice = X_test.sample(200, random_state=0)
sk = partial_dependence(model, X_ice, ["MedInc"], kind="individual", grid_resolution=20, percentiles=(0.05, 0.95))
grid = sk["grid_values"][0]
ice = ice_curves(model, X_ice, "MedInc", grid)
print("Largest difference from sklearn:", np.abs(ice - sk["individual"][0]).max())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(grid, ice[:30].T, color=BLUE, alpha=0.6, lw=1)
ax.scatter(np.full(30, grid[0]), ice[:30, 0], color=BLUE, s=12, zorder=3)
ax.set_xlabel("MedInc — median income of the block group, tens of thousands of dollars")
ax.set_ylabel("prediction, hundreds of thousands of dollars")
ax.set_title("ICE for MedInc on 30 California Housing objects, gradient boosting")
plt.show()

The curves start at different levels: every block group has its own prediction already at the lowest income. Let us draw all 200 curves and their mean over the objects — which is the PDP, the topic of the next lesson of the module.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(grid, ice.T, color=BLUE, alpha=0.2, lw=0.8)
ax.plot(grid, ice.mean(axis=0), color=ORANGE, lw=3, label="the mean of the ICE curves (PDP)")
ax.set_xlabel("MedInc — median income of the block group, tens of thousands of dollars")
ax.set_ylabel("prediction, hundreds of thousands of dollars")
ax.set_title("ICE for MedInc on 200 California Housing objects, gradient boosting")
ax.legend()
plt.show()

With thousands of objects the bundle merges into a solid band: individual trajectories can no longer be read, only the mean and the overall spread are visible. Let us draw the ICE for the feature `HouseAge` for every object of the test sample.

In [ ]:
g_age = quantile_grid(X_train, "HouseAge")
ice_all = ice_curves(model, X_test, "HouseAge", g_age)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(g_age, ice_all.T, color=BLUE, alpha=0.08, lw=0.6)
ax.plot(g_age, ice_all.mean(axis=0), color=ORANGE, lw=3, label="mean")
ax.set_xlabel("HouseAge — median house age in the block group, years")
ax.set_ylabel("prediction, hundreds of thousands of dollars")
ax.set_title(f"ICE for HouseAge on all {len(X_test)} test objects, gradient boosting")
ax.legend()
plt.show()

## 4. c-ICE

$$g_i^{c}(x_j) = g_i(x_j) - g_i(a),$$

where the anchor point $a$ is the first grid value.

**Task 4.** Complete `center` and draw c-ICE for four features: `MedInc`, `AveOccup`, `HouseAge`, `Latitude`. Draw the mean of the curves on top. Compute the spread of the centered curves at the last grid point — the standard deviation over the 200 objects.

In [ ]:
def center(ice):
    return ...  # your code: subtract from each curve its value at the first grid point


features = ["MedInc", "AveOccup", "HouseAge", "Latitude"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
spread_end = {}
for ax, feature in zip(axes.ravel(), features):
    g = quantile_grid(X_train, feature)
    c = center(ice_curves(model, X_ice, feature, g))
    spread_end[feature] = c[:, -1].std()
    ax.plot(g, c.T, color=BLUE, alpha=0.2, lw=0.8)
    ax.plot(g, c.mean(axis=0), color=ORANGE, lw=3)
    ax.set_xlabel(feature)
    ax.set_ylabel("prediction minus prediction at a")
fig.suptitle("c-ICE for 200 California Housing objects, gradient boosting; the orange line is the mean")
fig.tight_layout()
plt.show()
pd.Series(spread_end, name="c-ICE spread at the last point").round(3)

**Space for your conclusions:** for which of the four features do the centered curves diverge most, i.e. the feature's effect depends most on the other features of the block group? For which least?

## 5. d-ICE

A derivative of a boosting model cannot be taken: tree predictions are piecewise constant. So we compute the slope as a finite difference on the grid segments:

$$g_i'(x_j) \approx \frac{g_i(z_{l+1}) - g_i(z_l)}{z_{l+1} - z_l}.$$

**Task 5.** Complete `d_ice` and for each of the four features find the grid segment on which the slopes of different objects diverge most: the maximum of the standard deviation of the slopes over the objects.

In [ ]:
def d_ice(ice, grid):
    return ...  # your code: a (len(ice), len(grid) - 1) matrix of slopes


rows = {}
for feature in features:
    g = quantile_grid(X_train, feature)
    s = d_ice(ice_curves(model, X_ice, feature, g), g).std(axis=0)
    l = s.argmax()
    rows[feature] = {"largest slope spread": s.max(),
                     "segment from": g[l], "to": g[l + 1]}
pd.DataFrame(rows).T.round(3)

**Space for your conclusions:** on which segment and for which feature is the interaction with the other features strongest? Compare with the c-ICE from task 4.

## 6. ICE of a linear model

We train a linear regression on the same data and draw its ICE curves for the feature `HouseAge` for 40 objects.

**Task 6.** Check two things: the d-ICE slopes are the same for every object, and this common slope equals the weight of `HouseAge` in the model (`lin.coef_`).

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
g = quantile_grid(X_train, "HouseAge")
ice_lin = ice_curves(lin, X_ice.iloc[:40], "HouseAge", g)
slopes_lin = d_ice(ice_lin, g)
w = ...  # your code: the weight of HouseAge in the linear model
print("Spread of the slopes across objects:", slopes_lin.std(axis=0).max())
print("Slope:", slopes_lin[0, 0], " weight:", w)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(g, ice_lin.T, color=BLUE, alpha=0.5, lw=1)
ax.plot(g, ice_lin.mean(axis=0), color=ORANGE, lw=2.5, label="average")
ax.set_xlabel("HouseAge — median house age in the block group, years")
ax.set_ylabel("prediction, hundreds of thousands of dollars")
ax.set_title("ICE for HouseAge on 40 California Housing objects")
ax.legend()
plt.show()

**Space for your conclusions:** why are the ICE curves of a linear model parallel, and what follows from that for c-ICE and d-ICE?

## 7. Six plots of one model

Below are six plots of the same gradient boosting model on California Housing. Each one shows how the prediction depends on one feature, but they are drawn in six different ways. This is the picture of the last task of the lesson “ICE” on the course platform: decide for each plot whether it is an ICE plot.

In [ ]:
g_lat = quantile_grid(X_train, "Latitude")
ice_lat = ice_curves(model, X_ice, "Latitude", g_lat)
g_occ = quantile_grid(X_train, "AveOccup")
ice_occ = ice_curves(model, X_ice.iloc[:25], "AveOccup", g_occ)
g_rooms = quantile_grid(X_train, "AveRooms")
pdp_rooms = ice_curves(model, X_ice, "AveRooms", g_rooms).mean(axis=0)
c_med = center(ice[:60])
lat_bins = pd.cut(X_ice["Latitude"].iloc[:60], [32, 34, 37.5, 42], labels=["below 34", "34 to 37.5", "above 37.5"])

fig, axes = plt.subplots(3, 2, figsize=(12, 12))
ax = axes[0, 0]
for label, color in zip(lat_bins.cat.categories, [BLUE, AQUA, ORANGE]):
    ax.plot(grid, c_med[(lat_bins == label).to_numpy()].T, color=color, alpha=0.6, lw=1)
    ax.plot([], [], color=color, label=label)
ax.legend(title="Latitude")
ax.set_xlabel("MedInc")
ax.set_ylabel("prediction minus prediction at a")

axes[0, 1].plot(g_rooms, pdp_rooms, color=BLUE, lw=2)
axes[0, 1].set_xlabel("AveRooms")

m, s = ice.mean(axis=0), ice.std(axis=0)
axes[1, 0].fill_between(grid, m - s, m + s, color=BLUE, alpha=0.2, lw=0)
axes[1, 0].plot(grid, m, color=BLUE, lw=1.5)
axes[1, 0].set_xlabel("MedInc")

axes[1, 1].plot(g_lat, ice_lat.T, color=MUTED, alpha=0.4, lw=0.8)
axes[1, 1].set_xlabel("Latitude")

axes[2, 0].plot(g_age, ice_all.T, color=BLUE, alpha=0.08, lw=0.6)
axes[2, 0].plot(g_age, ice_all.mean(axis=0), color=ORANGE, lw=3)
axes[2, 0].set_xlabel("HouseAge")

axes[2, 1].plot(g_occ, ice_occ.T, lw=1.2)
axes[2, 1].set_xlabel("AveOccup")

for i, ax in enumerate(axes.ravel(), 1):
    ax.set_title(f"{i}.", loc="left", fontsize=16, fontweight="bold")
    if i > 1:
        ax.set_ylabel("prediction, hundreds of thousands of dollars")
fig.suptitle("California Housing, gradient boosting (HistGradientBoostingRegressor): six plots of the prediction against one feature")
fig.tight_layout()
plt.show()